# cGAN counterfactual explanation trên COVID-QU-Ex: COVID-19 vs Normal

- Dữ liệu: `dacntt-kltn/covid-normal-cxr` (train / validation / test), nhãn `normal=0`, `covid19=1` → **tải về thành file ảnh trên đĩa** 
- Luồng làm việc:
  1. Tải ảnh ra file và tạo `images.csv` (đường dẫn ảnh + nhãn)
  2. Tải DenseNet121 đã huấn luyện từ HF → chạy `compute_scores.py` → `scores_all.csv` (đường dẫn ảnh + điểm xác suất `score`)
  3. **Thống kê số ảnh trong từng bin** để chọn `PER_BIN` hợp lý
  4. Huấn luyện cGAN bằng `train.py`, **checkpoint đẩy lên Hugging Face** (đồng bộ nền + lần cuối, có resume)

## 1. Import

In [ ]:
import os, re, sys, io, json, time, shutil, threading, subprocess
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # tắt thanh tiến trình của HF
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

import torch

from datasets import load_dataset, Image as HFImage
from huggingface_hub import login, HfApi, create_repo, list_repo_files, hf_hub_download
from kaggle_secrets import UserSecretsClient
from IPython.display import display, Image as IPImage

## 2. Cấu hình

`PER_BIN`, `ITERS`, `BATCH_SIZE`, `GAN_SPLITS` ảnh hưởng đến phần thống kê ở mục 7. Sau khi xem thống kê có thể sửa ô này rồi chạy lại từ mục 7 trở đi.

In [ ]:
# Hugging Face
HF_USERNAME = "dacntt-kltn"
DATASET_REPO_ID = f"{HF_USERNAME}/covid-normal-cxr"
CLASSIFIER_REPO_ID = f"{HF_USERNAME}/densenet121-covid-normal-cxr"
CLASSIFIER_FILENAME = "training-runs/00000-densenet121-covid-normal/best.pt" 
MODEL_REPO_ID = f"{HF_USERNAME}/cgan-covid-normal-cxr"
MODEL_PRIVATE = False
RUNS_PREFIX = "training-runs"
RUN_SUFFIX = "cgan-covid-normal"

# Code trên GitHub
GITHUB_REPO_URL = "https://github.com/Vinh306/medical-image-classification-counterfactual-explanation"
CODE_DIR = Path("/kaggle/working/cf-explanation-code")
CODE_SUBDIR = "cGAN"

# Dữ liệu trên đĩa 
DATA_DIR = Path("/tmp/covid-normal-cxr")
IMAGES_DIR = DATA_DIR / "images"
IMAGES_CSV = DATA_DIR / "images.csv" # path, split, label, label_name, covid19
SCORES_ALL_CSV = DATA_DIR / "scores_all.csv" # + cột score
TRAIN_CSV = DATA_DIR / "scores_gan_train.csv" # các split dùng để train cGAN
FORCE_RECOMPUTE_SCORES = False

TARGET = "covid19"
GAN_SPLITS = ["train"] # split dùng để train cGAN; có thể thêm "validation" nếu bin quá thưa

# Tham số cGAN (mặc định của train.py)
SEED = 42
NUM_WORKERS = 2
ITERS = 50000 # tổng số bước D; G/E cập nhật ITERS / N_CRITIC lần
BATCH_SIZE = 16
PER_BIN = 3000 # số ảnh tối đa mỗi bin
LR = 2e-4
N_CRITIC = 5
LAM_GAN, LAM_F, LAM_REC = 1.0, 1.0, 0.5
LOG_EVERY = 50 # tính theo số bước G
SAVE_EVERY = 5000
SAMPLE_EVERY = 2500
USE_AMP = False

# Đồng bộ lên Hub
RESUME_FROM_HUB = False  # True: nếu run mới nhất trên Hub chưa xong thì train tiếp
SYNC_INTERVAL_SEC = 300  # đẩy checkpoint lên Hub mỗi 5 phút
PRUNE_LOCAL_MILESTONES = True # xóa checkpoint mốc cũ trên đĩa SAU KHI đã lên Hub

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| AMP:", USE_AMP)

## 3. Lấy code từ GitHub

In [ ]:
if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(CODE_DIR)], check=True)

GIT_COMMIT = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"],
                            capture_output=True, text=True, check=True).stdout.strip()
print("Commit đang dùng:", GIT_COMMIT)

CODE_PATH = CODE_DIR / CODE_SUBDIR
sys.path.insert(0, str(CODE_PATH))
import common, data, models, classifier, losses
for m in (common, data, models, classifier, losses):
    assert str(CODE_DIR) in m.__file__, f"Import nhầm module trùng tên: {m.__name__} -> {m.__file__}"

from common import N_BINS, IMG_SIZE, LABELS, seed_everything
from data import load_image, score_to_bin_np, balance_by_bin
assert LABELS == [TARGET], f"Repo cần LABELS = ['{TARGET}'] (1 logit), hiện tại: {LABELS}"

seed_everything(SEED)
print("Đã import từ", CODE_PATH, "| IMG_SIZE =", IMG_SIZE, "| N_BINS =", N_BINS)

# Chạy script của repo và in log theo thời gian thực
def run_cmd(cmd, cwd=None, check=True):
    cmd = [str(c) for c in cmd]
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    except BaseException:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    if check and rc != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {rc}): {' '.join(cmd)}")
    return rc

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

## 4. Đăng nhập Hugging Face và tạo repo model

In [ ]:
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

api = HfApi()
print("Đăng nhập là:", api.whoami()["name"])
create_repo(MODEL_REPO_ID, repo_type="model", private=MODEL_PRIVATE, exist_ok=True)
print(f"Model repo: https://huggingface.co/{MODEL_REPO_ID}")
print(f"Dataset repo: https://huggingface.co/datasets/{DATASET_REPO_ID}")

## 5. Tải dữ liệu về file cứng và tạo CSV đường dẫn ảnh

In [ ]:
ds = load_dataset(DATASET_REPO_ID, token=token)
print(ds)

label_names = ds["train"].features["label"].names
print("Nhãn:", label_names)
assert label_names == ["normal", "covid19"], "Thứ tự nhãn không đúng (cần normal=0, covid19=1)"
covid_idx = label_names.index("covid19")

def ext_of(b):
    if b[:8] == b"\x89PNG\r\n\x1a\n": return ".png"
    if b[:2] == b"\xff\xd8": return ".jpg"
    return ".png"

rows, n_written = [], 0
for split in ds:
    d = ds[split].cast_column("image", HFImage(decode=False)) # lấy bytes gốc
    for i, ex in enumerate(tqdm(d, desc=split)):
        b = ex["image"]["bytes"]
        assert b is not None, f"{split}[{i}] không có bytes ảnh"
        name = label_names[ex["label"]]
        p = IMAGES_DIR / split / name / f"{i:06d}{ext_of(b)}"
        if not p.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
            tmp = p.with_suffix(p.suffix + ".tmp")
            tmp.write_bytes(b)
            os.replace(tmp, p)
            n_written += 1
        rows.append(dict(path=str(p), split=split, label=int(ex["label"]), label_name=name,
                         covid19=float(ex["label"] == covid_idx)))

images_df = pd.DataFrame(rows)
images_df.to_csv(IMAGES_CSV, index=False)
print(f"\nGhi mới {n_written} file | tổng {len(images_df)} ảnh | CSV: {IMAGES_CSV}")
print(images_df.groupby(["split", "label_name"]).size().unstack(fill_value=0))

# Kiểm tra ảnh đọc được bằng đúng hàm load_image của repo
im = PILImage.open(images_df["path"].iloc[0])
print("Ảnh mẫu:", im.size, im.mode)
if im.size != (IMG_SIZE, IMG_SIZE):
    print(f"CẢNH BÁO: ảnh {im.size} khác {IMG_SIZE}x{IMG_SIZE}; load_image của repo sẽ tự crop giữa và resize.")
for p in images_df["path"].sample(8, random_state=SEED):
    t = load_image(p)
    assert t.shape == (1, IMG_SIZE, IMG_SIZE) and 0 <= t.min() and t.max() <= 1
print("load_image OK:", tuple(t.shape))

## 6. Tính điểm xác suất của classifier cho từng ảnh

In [ ]:
# Tra cứu hướng dẫn và danh sách tham số của compute_scores.py
run_cmd([sys.executable, "compute_scores.py", "--help"], cwd=CODE_PATH)

In [ ]:
clf_file = CLASSIFIER_FILENAME
if not clf_file:
    files = list_repo_files(CLASSIFIER_REPO_ID, repo_type="model")
    clf_file = sorted(f for f in files if f.startswith(f"{RUNS_PREFIX}/") and f.endswith("/best.pt"))[-1]
print("Classifier:", f"{CLASSIFIER_REPO_ID}/{clf_file}")

CLASSIFIER_PATH = DATA_DIR / "classifier_best.pt"
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download(CLASSIFIER_REPO_ID, clf_file, repo_type="model"), CLASSIFIER_PATH)
ck = torch.load(CLASSIFIER_PATH, map_location="cpu", weights_only=False)
print(f"best.pt: epoch {ck.get('epoch')}, {ck.get('monitor')} = {ck.get('value')}")
del ck

need = FORCE_RECOMPUTE_SCORES or not SCORES_ALL_CSV.exists()
if not need:
    need = len(pd.read_csv(SCORES_ALL_CSV, usecols=["path"])) != len(images_df)
if need:
    run_cmd([sys.executable, "compute_scores.py",
             "--csv_path", IMAGES_CSV, 
             "--classifier_path", CLASSIFIER_PATH, 
             "--target", TARGET,
             "--output_path", SCORES_ALL_CSV, 
             "--batch_size", 64, 
             "--num_workers", NUM_WORKERS],
            cwd=CODE_PATH)
else:
    print(f"Đã có {SCORES_ALL_CSV}, bỏ qua tính điểm (đặt FORCE_RECOMPUTE_SCORES = True để tính lại).")

sc = pd.read_csv(SCORES_ALL_CSV)
assert {"path", "score", "split", "label"} <= set(sc.columns) and sc["score"].between(0, 1).all()

for s in sc["split"].unique():
    sc[sc.split == s].to_csv(DATA_DIR / f"scores_{s}.csv", index=False)
sc[sc.split.isin(GAN_SPLITS)].to_csv(TRAIN_CSV, index=False)
print(f"\nscores_all.csv: {len(sc)} ảnh | CSV train cGAN ({GAN_SPLITS}): {int(sc.split.isin(GAN_SPLITS).sum())} ảnh -> {TRAIN_CSV}")

## 7. Thống kê và phân tích số ảnh theo bin

In [ ]:
SPARSE_THRESHOLD = 100   # bin có ít hơn ngần này ảnh được coi là thưa

sc = pd.read_csv(SCORES_ALL_CSV)
sc["bin"] = score_to_bin_np(sc["score"].values)
splits = list(sc["split"].unique())

print("AUC của classifier theo split:")
for s in splits:
    g = sc[sc.split == s]
    print(f"  {s:>10}: AUC = {roc_auc_score(g['label'], g['score']):.4f} | n = {len(g)}")

# Số ảnh mỗi bin theo từng split
bin_names = [f"[{b / N_BINS:.1f}, {(b + 1) / N_BINS:.1f})" for b in range(N_BINS)]
bin_names[-1] = bin_names[-1][:-1] + "]"
cnt = pd.crosstab(sc["bin"], sc["split"]).reindex(index=range(N_BINS), columns=splits, fill_value=0)
cnt["all"] = cnt.sum(axis=1)
pct = (cnt / cnt.sum() * 100).round(1)
show = cnt.astype(str) + " (" + pct.astype(str) + "%)"
show.index = bin_names
print("\n(a) Số ảnh mỗi bin theo split:")
print(show.to_string())

# Bin của các split dùng để train cGAN, tách theo nhãn thật
tr = sc[sc.split.isin(GAN_SPLITS)]
by_label = pd.crosstab(tr["bin"], tr["label_name"]).reindex(index=range(N_BINS), columns=label_names, fill_value=0)
by_label["total"] = by_label.sum(axis=1)
by_label.index = bin_names
print(f"\n(b) Split dùng train cGAN {GAN_SPLITS} - số ảnh mỗi bin theo nhãn thật:")
print(by_label.to_string())

ct = by_label["total"].values
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
x, bottom = np.arange(N_BINS), np.zeros(N_BINS)
for name in label_names:
    ax[0].bar(x, by_label[name].values, bottom=bottom, label=name); bottom += by_label[name].values
ax[0].set_title(f"Số ảnh mỗi bin ({'+'.join(GAN_SPLITS)}), tách theo nhãn thật"); ax[0].legend()
ax[1].bar(x, ct, color="tab:gray"); ax[1].set_yscale("symlog")
ax[1].axhline(PER_BIN, color="red", ls="--", label=f"PER_BIN = {PER_BIN}")
ax[1].set_title("Cùng dữ liệu, thang symlog"); ax[1].legend()
for a in ax:
    a.set_xticks(x); a.set_xticklabels(bin_names, rotation=45, ha="right"); a.set_xlabel("bin của score"); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(DATA_DIR / "bin_distribution.png", dpi=300); plt.show()

# Chọn PER_BIN: so sánh các giá trị ứng viên
cand = []
for pb in sorted({200, 500, 1000, 2000, 3000, 5000, PER_BIN}):
    eff = np.minimum(ct, pb)
    cand.append(dict(per_bin=pb, tong_anh=int(eff.sum()), bin_du_PER_BIN=int((ct >= pb).sum()),
                     bin_rong=int((ct == 0).sum()), bin_it_nhat=int(eff[eff > 0].min()) if (eff > 0).any() else 0))
print("\n(c) Ảnh dùng được theo từng giá trị PER_BIN:")
print(pd.DataFrame(cand).to_string(index=False))

# Với cấu hình hiện tại
bal = balance_by_bin(tr, PER_BIN, SEED)
eff_ct = np.bincount(score_to_bin_np(bal["score"].values), minlength=N_BINS)
draws_per_bin = ITERS * BATCH_SIZE / N_BINS    # sampler lấy đều các bin
cur = pd.DataFrame(dict(bin=bin_names, anh_goc=ct, anh_sau_balance=eff_ct,
                        so_lan_lap=np.where(eff_ct > 0, draws_per_bin / np.maximum(eff_ct, 1), np.nan).round(1)))
print(f"\n(d) Cấu hình hiện tại: PER_BIN={PER_BIN}, ITERS={ITERS}, BATCH_SIZE={BATCH_SIZE} "
      f"-> mỗi bin được lấy ~{draws_per_bin:.0f} lượt ảnh trong cả quá trình train")
print(cur.to_string(index=False))

stats_df = pd.concat([cnt.set_axis(bin_names), by_label.add_prefix("gan_"), cur.set_index("bin")[["anh_sau_balance", "so_lan_lap"]]], axis=1)
stats_df.to_csv(DATA_DIR / "bin_stats.csv")

## 8. Kích thước mô hình và checkpoint (ước tính)

In [ ]:
from models import Explainer, Discriminator
n_x = sum(p.numel() for p in Explainer().parameters())
n_d = sum(p.numel() for p in Discriminator().parameters())
ckpt_gb = (n_x + n_d) * 4 * 3 / 1e9    # trọng số + 2 moment của Adam, fp32
n_ms = ITERS // SAVE_EVERY
print(f"Explainer (E+G): {n_x / 1e6:.1f}M tham số | Discriminator: {n_d / 1e6:.1f}M tham số")
print(f"Mỗi checkpoint ~{ckpt_gb:.1f} GB (ước tính) | {n_ms} mốc -> ~{ckpt_gb * n_ms:.0f} GB nếu giữ hết trên đĩa")

## 9. Thư mục run và resume từ Hub

In [ ]:
LOCAL_OUTDIR = Path("/kaggle/working/cgan-training-runs")
LOCAL_OUTDIR.mkdir(parents=True, exist_ok=True)

existing_files = list_repo_files(MODEL_REPO_ID, repo_type="model")
run_re = re.compile(rf"^{RUNS_PREFIX}/([0-9]{{5}}-[^/]+)/(.+)$")
hub_runs = {}
for f in existing_files:
    m = run_re.match(f)
    if m:
        hub_runs.setdefault(m.group(1), set()).add(m.group(2))
print("Các run đã có trên Hub:", sorted(hub_runs))

RESUME_CKPT, RUN_NAME = None, None
if RESUME_FROM_HUB and hub_runs:
    latest = sorted(hub_runs)[-1]
    if "explainer_last.pt" in hub_runs[latest] and "finished.json" not in hub_runs[latest]:
        RUN_NAME = latest
        RUN_DIR = LOCAL_OUTDIR / RUN_NAME
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        for fn in ["explainer_last.pt", "history.csv", "train.log", "config.json", "run_info.json"]:
            if fn in hub_runs[latest]:
                p = hf_hub_download(MODEL_REPO_ID, f"{RUNS_PREFIX}/{latest}/{fn}", repo_type="model")
                shutil.copy(p, RUN_DIR / fn)
        RESUME_CKPT = RUN_DIR / "explainer_last.pt"
        it_done = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)["it"]
        print(f"RESUME run {RUN_NAME} từ iter {it_done}")

if RESUME_CKPT is None:
    next_id = int(sorted(hub_runs)[-1][:5]) + 1 if hub_runs else 0
    RUN_NAME = f"{next_id:05d}-{RUN_SUFFIX}"
    RUN_DIR = LOCAL_OUTDIR / RUN_NAME
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "run_info.json").write_text(json.dumps(dict(
        model="cGAN explainer (Singla et al., 2022)", dataset=DATASET_REPO_ID, target=TARGET,
        classifier=f"{CLASSIFIER_REPO_ID}/{clf_file}", gan_splits=GAN_SPLITS,
        iters=ITERS, batch_size=BATCH_SIZE, per_bin=PER_BIN, lr=LR, n_critic=N_CRITIC,
        lam_gan=LAM_GAN, lam_f=LAM_F, lam_rec=LAM_REC, use_amp=USE_AMP, seed=SEED,
        code_repo=GITHUB_REPO_URL, code_commit=GIT_COMMIT), indent=2))
    print("Train từ đầu, run mới:", RUN_NAME)

# Lưu CSV điểm số và thống kê bin cùng run để tái lập
(RUN_DIR / "data_info").mkdir(exist_ok=True)
for src in [SCORES_ALL_CSV, TRAIN_CSV, DATA_DIR / "bin_stats.csv", DATA_DIR / "bin_distribution.png"]:
    shutil.copy(src, RUN_DIR / "data_info" / src.name)

## 10. Huấn luyện cGAN + đồng bộ Hub

In [ ]:
# Tra cứu hướng dẫn và danh sách toàn bộ các tham số của train.py
run_cmd([sys.executable, "train.py", "--help"], cwd=CODE_PATH)

In [ ]:
api_sync = HfApi()
sync_stop = threading.Event()

def upload_run_dir(msg):
    api_sync.upload_folder(folder_path=str(RUN_DIR), repo_id=MODEL_REPO_ID, repo_type="model",
                           path_in_repo=f"{RUNS_PREFIX}/{RUN_NAME}", ignore_patterns=["*.tmp"],
                           commit_message=msg)

def dir_signature():
    return tuple(sorted((str(p.relative_to(RUN_DIR)), p.stat().st_mtime) for p in RUN_DIR.rglob("*")
                        if p.is_file() and p.suffix != ".tmp"))

def prune_milestones(uploaded_names):
    # Chỉ xóa mốc cũ đã nằm trong lần upload vừa xong, luôn giữ mốc mới nhất
    ms = sorted(RUN_DIR.glob("explainer_it*.pt"))
    for p in ms[:-1]:
        if p.name in uploaded_names:
            p.unlink()
            print(f"[sync] đã xóa {p.name} trên đĩa (vẫn còn trên Hub).")

def sync_to_hub():
    last_sig = None
    while not sync_stop.wait(SYNC_INTERVAL_SEC):
        try:
            sig = dir_signature()
            if sig and sig != last_sig:
                snapshot = {p.name for p in RUN_DIR.glob("explainer_it*.pt")}
                upload_run_dir(f"sync {RUN_NAME}")
                print("[sync] đã đồng bộ lên Hub.")
                if PRUNE_LOCAL_MILESTONES:
                    prune_milestones(snapshot)
                last_sig = dir_signature()
        except Exception as e:
            print("[sync] Lỗi khi đồng bộ:", e)

sync_thread = threading.Thread(target=sync_to_hub, daemon=True)
sync_thread.start()

cmd = [sys.executable, "-u", "train.py",
       "--csv_path", TRAIN_CSV, 
       "--classifier_path", CLASSIFIER_PATH, 
       "--target", TARGET,
       "--out_dir", RUN_DIR, 
       "--iters", ITERS, 
       "--batch_size", BATCH_SIZE, 
       "--per_bin", PER_BIN,
       "--lr", LR, 
       "--n_critic", N_CRITIC, 
       "--lam_gan", LAM_GAN, 
       "--lam_f", LAM_F, 
       "--lam_rec", LAM_REC,
       "--log_every", LOG_EVERY, 
       "--save_every", SAVE_EVERY, 
       "--sample_every", SAMPLE_EVERY,
       "--num_workers", NUM_WORKERS, 
       "--seed", SEED]
if USE_AMP:
    cmd.append("--use_amp")
if RESUME_CKPT is not None:
    cmd += ["--resume", RESUME_CKPT]
print("Lệnh:", " ".join(str(c) for c in cmd), "\n")

rc = None
try:
    rc = run_cmd(cmd, cwd=CODE_PATH, check=False)
    if rc == 0:   # chỉ đánh dấu đã xong khi train.py kết thúc bình thường
        (RUN_DIR / "finished.json").write_text(json.dumps(dict(
            reason="max_iters", iters=ITERS, last_checkpoint="explainer_last.pt"), indent=2))
finally:
    sync_stop.set()
    sync_thread.join(timeout=600)
    upload_run_dir(f"final {RUN_NAME}")
    print(f"Đã đẩy kết quả lên Hugging Face: https://huggingface.co/{MODEL_REPO_ID}/tree/main/{RUNS_PREFIX}/{RUN_NAME}")

if rc != 0:
    raise RuntimeError(f"train.py kết thúc với mã lỗi {rc}; checkpoint gần nhất đã được đẩy lên Hub, đặt RESUME_FROM_HUB = True để train tiếp.")

## 11. Đường cong loss và ảnh phản thực tế mẫu

In [ ]:
curves = RUN_DIR / "loss_curves.png"
if curves.exists():
    display(IPImage(filename=str(curves)))

samples = sorted(RUN_DIR.glob("sample_it*_*.png"))
if samples:
    last_it = samples[-1].name.split("_")[1]  
    print("Ảnh mẫu của lần sinh gần nhất:", last_it)
    for p in [s for s in samples if s.name.split("_")[1] == last_it]:
        display(IPImage(filename=str(p)))

## 12. Đẩy kết quả cuối lên Hub

In [ ]:
upload_run_dir(f"final results {RUN_NAME}")
print(f"Xong: https://huggingface.co/{MODEL_REPO_ID}/tree/main/{RUNS_PREFIX}/{RUN_NAME}")

## 13. Sinh ảnh phản thực tế từ một checkpoint bất kỳ trên Hub

In [ ]:
HF_CHECKPOINT_FILENAME = "" # ví dụ: "training-runs/00000-cgan-covid-normal/explainer_it10000.pt"
GEN_GROUP, GEN_N = "any", 8 # "neg" | "mild" | "pos" | "any"

if not HF_CHECKPOINT_FILENAME:
    files = list_repo_files(MODEL_REPO_ID, repo_type="model")
    HF_CHECKPOINT_FILENAME = sorted(f for f in files if f.startswith(f"{RUNS_PREFIX}/") and f.endswith("/explainer_last.pt"))[-1]
print("Dùng checkpoint:", HF_CHECKPOINT_FILENAME)

hub_ckpt = DATA_DIR / "hub_explainer.pt"
shutil.copy(hf_hub_download(MODEL_REPO_ID, HF_CHECKPOINT_FILENAME, repo_type="model"), hub_ckpt)

GEN_CSV = DATA_DIR / "scores_test.csv"
if not GEN_CSV.exists():
    GEN_CSV = TRAIN_CSV
GEN_DIR = Path("/kaggle/working/hub_vis")
run_cmd([sys.executable, "generate.py", 
        "--csv_path", GEN_CSV, 
        "--classifier_path", CLASSIFIER_PATH,
        "--explainer_path", hub_ckpt, 
        "--target", TARGET, 
        "--group", GEN_GROUP, 
        "--n", GEN_N,
        "--out_dir", GEN_DIR, 
        "--seed", SEED], 
        cwd=CODE_PATH)
for p in sorted(GEN_DIR.glob("*.png")):
    display(IPImage(filename=str(p)))